In [1]:
import os
# make sure Pandeia is set up correctly
os.environ["pandeia_refdata"] = "/Users/jillianpaulin/pandeia_jwst_data/pandeia_data-2025.7-jwst"
os.environ["PYSYN_CDBS"] = "/Users/jillianpaulin/pandeia_jwst_data/pandeia_data-2025.7-jwst/grp/redcat/trds"

from copy import deepcopy
import synphot
from astropy import units as u
from matplotlib import pyplot as plt
from pandeia.engine.calc_utils import build_default_calc
from pandeia.engine.perform_calculation import perform_calculation
import pandas as pd
import scipy.integrate as integrate
import math
import numpy as np
from astropy.io import fits
from matplotlib.ticker import AutoMinorLocator
from __future__ import print_function 
from scipy.optimize import curve_fit 
import scipy.optimize

c=3.0*1e10
H_0=7.2*1e6
h=6.63*1e-27
k_b=1.38*1e-16
D_H=c/H_0*3.086*1e24
Omega_M=0.27
Omega_Alpha=0.73
parsec=3.086e+18

In [2]:
# SMDS functions

# functions to get the redshifted spectra for SMDS
# convert individual string 'x.xxxD+xx' to number
def strtonum(strD):
    dot=1  #position of dot
    digit=99999 #position of D
    Num=0.0 #converted number
    sign=0 #position of the sign
    for i in range(len(strD)):
        if strD[i]=='+':
            sign=1;
        if strD[i]=='-':
            sign=-1
        if i==0:
            Num+=int(float(strD[i]))
        if strD[i]=='D' or strD[i]=='E':
            digit=i
        if sign==0:
            if i>dot and i<digit:
                Num+=int(strD[i])*10**(-(i-1))
        elif i>digit+1 and sign==1: 
                Num*=10**(int(strD[i])*10**(len(strD)-i-1)) 
        elif i>digit+1 and sign==-1:
                Num*=10**(-int(strD[i])*10**(len(strD)-i-1))
    return Num

# convert the whole array of such string
def convertdata(data):
    array=[]
    for i in range(len(data)):
        array.insert(i,strtonum(data[i]))
    return array

#convert from frequency to wavelength
def freqtolam_ETC(data,z):
    freq=convertdata(data[0:,0])
    Lambda_obs=[]
    Lambda_obs.insert(1,(c/freq[1])*(1+z)*1e4)
    lam_i=Lambda_obs[0]
    for i in range(len(freq)-1):
        lam_i=c/freq[i+1]*(1+z)*1e4
#get rid of same data pts
        if lam_i==Lambda_obs[i]:
            Lambda_obs.insert(i+1,lam_i+1e-7)
        else:
            Lambda_obs.insert(i+1,lam_i)
    return Lambda_obs
    
#shift the spectrum to mJy
def shftspecTLUmJy(data,z,R,mu):
    freq=convertdata(data[0:,0])
    flux=convertdata(data[0:,1])
    #eddfactor=data[0:,2]
    I = integrate.quad(lambda x: D_H*1/math.sqrt(Omega_M*(1+x)**3+Omega_Alpha), 0, z)
    DC_z=I[0]-I[1]
    DL_z=(1+z)*DC_z
    fluxobs=[]
    lam_lyman=(z+1)*0.1216
    lam=freqtolam_ETC(data,z)
    for i in range(len(lam)):
        if lam[i]<lam_lyman:
            fluxobs.insert(i,0)
        else:
            fluxobs.insert(i,4*np.pi*flux[i]*(1+z)*R**2/DL_z**2*1e26*mu)
    #flux=list(map(lambda x: ,Lambda))
    return fluxobs


In [ ]:
def save_as_fits(data, filename):
    hdu = fits.PrimaryHDU(data)
    hdul = fits.HDUList([hdu])
    hdul.writeto(filename, overwrite=True)
#     print(f"Saved FITS file: {filename}")
def generate_smds_profile(major, minor, sersic_index, candidate, shape):
    telescope = 'jwst'
    instrument = 'nircam'
    mode = 'sw_imaging'
    configuration = build_default_calc(telescope, instrument, mode)
    configuration['configuration']['instrument']['aperture'] = "sw"
    configuration['configuration']['instrument']['filter'] = "f200w"
    if candidate == 'JZ11':
        configuration['configuration']['detector']['ngroup'] = 5 # groups per integration
        configuration['configuration']['detector']['nint'] = 1 # integrations per exposure
        configuration['configuration']['detector']['nexp'] = 35 # exposures
        BestFitz11 = pd.read_csv("/Users/jillianpaulin/Desktop/ResearchF22/SMDS_Best_Fit_Jun2024/T3M6_0e5.13"
                         ,header=None,sep='\s+')
        data = BestFitz11.to_numpy()
        SMDS_lam=freqtolam_ETC(data, 11.37)
        SMDS_flux=shftspecTLUmJy(data, 11.37,8.35e+14,1)
        filename = 'JZ11_f200w_sim.fits'
    elif candidate == 'JZ13':
        configuration['configuration']['detector']['ngroup'] = 5 # groups per integration
        configuration['configuration']['detector']['nint'] = 1 # integrations per exposure
        configuration['configuration']['detector']['nexp'] = 35 # exposures
        BestFitz13 = pd.read_csv("/Users/jillianpaulin/Desktop/ResearchF22/SMDS_Best_Fit_Jun2024/T3M4_0e5.13"
                         ,header=None,sep='\s+')
        data = BestFitz13.to_numpy()
        SMDS_lam=freqtolam_ETC(data, 13.17)
        SMDS_flux=shftspecTLUmJy(data, 13.17,7.632e+14,1)
        filename = 'JZ13_f200w_sim.fits'
    elif candidate == 'JZ14_0':
        configuration['configuration']['detector']['ngroup'] = 5 # groups per integration
        configuration['configuration']['detector']['nint'] = 1 # integrations per exposure
        configuration['configuration']['detector']['nexp'] = 35 # exposures
        BestFitz14_0 = pd.read_csv("/Users/jillianpaulin/Desktop/ResearchF22/SMDS_Best_Fit_Jun2024/ACM2_00e+06.13"
                         ,header=None,sep='\s+')
        data = BestFitz14_0.to_numpy()
        SMDS_lam=freqtolam_ETC(data, 14.44)
        SMDS_flux=shftspecTLUmJy(data, 14.44,1.01e+15,1.2)
        filename = 'JZ14_0_f200w_sim.fits'
    elif candidate == 'Maisies': # TODO: finish
        configuration['configuration']['detector']['ngroup'] = 7
        configuration['configuration']['detector']['nint'] = 1
        configuration['configuration']['detector']['nexp'] = 19
        BestFitMaisies = pd.read_csv("/Users/jillianpaulin/Desktop/ResearchF22/T3M9_0e5.13", header=None, sep='\s+')
        data = BestFitMaisies.to_numpy()
        SMDS_lam = freqtolam_ETC(data, 11.4)
        SMDS_flux=shftspecTLUmJy(data, 11.4, 8.935e+14, 1)
        filename = 'Maisies_f200w_sim.fits'
    
    configuration['configuration']['detector']['readout_pattern'] = "deep8"
    configuration['strategy']['target_xy'] = [0.0, 0.0]
    configuration['strategy']['aperture_size'] = 0.05
    configuration['strategy']['sky_annulus'] = [0.4, 0.6]
    configuration['strategy']['background_subtraction'] = True
    configuration['calculation']['noise']['crs'] = True
    configuration['calculation']['noise']['ffnoise'] = True
    custom_galaxy_name = 'My Background Galaxy'
    custom_galaxy_redshift = 0.001 # it gets angry at z=0
    custom_galaxy_extinction = 0 # magnitudes
    
    source_gal = deepcopy(configuration['scene'][0])
    source_gal['position']['orientation'] = 0
    # source_gal['shape']['geometry'] = 'point'
    
    if shape == 'sersic':
        source_gal['shape'] = {
            'geometry': 'sersic',
            'shape_parameters': ['major', 'minor', 'sersic_index'],
            'major': major,                    # arcseconds
            'minor': minor,                    # arcseconds
            'sersic_index': sersic_index,
            'norm_method': 'integ_infinity',
            'surf_area_units': 'arcsec^2'
        }
    elif shape == 'gaussian2d':
        source_gal['shape'] = {
            'geometry': 'gaussian2d',
            'shape_parameters': ['major', 'minor'],
            'major': major, 
            'minor': minor,
            'norm_method': 'integ_infinity',
            'surf_area_units': 'arcsec^2'
        }
    elif shape == 'point':
        filename = f'{candidate}_f200w_psf.fits'
        
    source_gal['spectrum']['name'] = custom_galaxy_name
    source_gal['spectrum']['redshift'] = custom_galaxy_redshift
    source_gal['spectrum']['extinction']['value'] = custom_galaxy_extinction
    source_gal['spectrum']['sed'] = {'sed_type': 'input', 'spectrum': [SMDS_lam, SMDS_flux]}
    source_gal['spectrum']['normalization'] = {}
    source_gal['spectrum']['normalization']['type'] = 'photsys'#'none' #'photsys'
    
    configuration['background']= "minzodi"
    configuration['background_level']="low"
    configuration['configuration']['instrument']['aperture'] = "sw"
    configuration['configuration']['instrument']['filter'] = "f200w"
    configuration['configuration']['detector']['readout_pattern'] = "deep8" # from Maisie's Galaxy data
    
    configuration['strategy']['target_xy'] = [0.0,0.0] # x,y location to extract, in arcsec
    configuration['strategy']['aperture_size'] = 0.2 # radius of extraction aperture, in arcsec
    configuration['strategy']['sky_annulus'] = [0.4,0.6] # inner and outer radii of background subtraction annulus, in arcsec
    configuration['strategy']['background_subtraction'] = True
    configuration['calculation']['noise']['crs']= True
    configuration['calculation']['noise']['ffnoise'] = True
    configuration['scene'] = [source_gal]
 
    result = perform_calculation(configuration)
    
    # Extract the 2D image data from the results
    image_data = result['2d']['detector']

    # Save the image data as a FITS file
    save_as_fits(image_data, filename)
    
    return()

In [4]:
def radial_profile(data, center, distances):
    x, y = np.indices((data.shape))
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2)
    # Create an empty array to hold the radial profile values at specific distances
    radialprofile = np.zeros(len(distances))
    for i, d in enumerate(distances):
        if i == 0:
            mask = (r < (distances[i] + distances[i+1]) / 2)  # First bin
        elif i == len(distances) - 1:
            mask = (r >= (distances[i-1] + distances[i]) / 2) & (r < distances[i] + (distances[i] - distances[i-1]) / 2)

        else:
            # Non-overlapping bin between two adjacent points
            lower_edge = (distances[i - 1] + distances[i]) / 2
            upper_edge = (distances[i] + distances[i + 1]) / 2
            mask = (r >= lower_edge) & (r < upper_edge)

        if np.sum(mask) > 0:
            radialprofile[i] = data[mask].mean()
        else:
            radialprofile[i] = 0
    return radialprofile

In [5]:
from sklearn.metrics import mean_squared_error
import seaborn as sns
def sampler_obs(obs_data, unc):
    n_sim = 75
    main_array = []
    for i in range(n_sim):
        sub_array = []
        for j in range(len(obs_data)):
            upper = obs_data[j] + unc[j]
            lower = obs_data[j] - unc[j]
            mean = obs_data[j]
            sigma = (upper - lower)/3.92
            sample = np.random.normal(mean, sigma)
            sub_array.append(sample)
        main_array.append(sub_array)

    return main_array

In [9]:
datafile_path = 'JZ13_rad_prof_data.csv'
data = pd.read_csv(datafile_path, header=None)
xdat = data.iloc[:, 0].values  # First column
ydat = data.iloc[:, 1].values  # Second column
lower_error = [0.008, 0.008, ydat[2]-0.18, ydat[3]-0.086, ydat[4]-0.038, ydat[5]-0.0067, ydat[6]-0.0011, ydat[7]-0.0018]
upper_error = [0.008, 0.008, 0.22-ydat[2], 0.12-ydat[3], 0.068-ydat[4], 0.034-ydat[5], 0.024-ydat[6], 0.026-ydat[7]]

sampler_obs(ydat, lower_error)
main_array = np.array(sampler_obs(ydat, lower_error))
print(main_array)

[[ 9.92669657e-01  5.21070856e-01  2.07740763e-01  1.17269373e-01
   5.61596356e-02  3.43017791e-02  7.03677916e-03  6.23864361e-03]
 [ 9.82319825e-01  5.30941099e-01  2.05630600e-01  1.03960307e-01
   5.51384578e-02  1.27034435e-02  2.31500239e-02  1.47634014e-02]
 [ 9.90026417e-01  5.27797108e-01  2.03931056e-01  1.06595942e-01
   4.69482790e-02  2.41919010e-02  2.03286657e-02  2.23097897e-02]
 [ 9.96082710e-01  5.22748201e-01  1.89513517e-01  1.03298238e-01
   5.34227233e-02  2.86717503e-02  2.43896809e-03  2.34522990e-02]
 [ 9.88934977e-01  5.23092424e-01  1.89292936e-01  8.88777692e-02
   5.92946070e-02  2.13276206e-02  4.45342939e-03  5.92576748e-03]
 [ 9.88209357e-01  5.31647129e-01  1.96670390e-01  9.03085627e-02
   5.11161302e-02  1.88382095e-02 -2.58798974e-03  1.36247033e-02]
 [ 9.88637317e-01  5.27676454e-01  2.17342072e-01  1.15109015e-01
   5.09390383e-02  1.28398637e-02  2.37886789e-02  1.43290416e-02]
 [ 9.89636762e-01  5.24736326e-01  2.01157334e-01  1.03123074e-01
   

In [7]:
def radprof_opt(x, array_obs, datafile, candidate, shape):
    major = x[0]
    minor = x[0]
    sersic_index = x[1]
    
    generate_smds_profile(major, minor, sersic_index, candidate, shape)
    simfile = f'{candidate}_f200w_sim.fits'
    psffile = f'{candidate}_f200w_psf.fits'
    # Open FITS files
    simFile = fits.open(simfile)
    simimg = simFile[0].data
    
    psfFile = fits.open(psffile)
    psfimg = psfFile[0].data

    # Find centers
    simcenter = np.unravel_index(simimg.argmax(), simimg.shape)
    psfcenter = np.unravel_index(psfimg.argmax(), psfimg.shape)

    # Read the CSV file for comparison points
    df = pd.read_csv(datafile, header=None)
    xpos = df.iloc[:, 0].values  # First column

    # Convert the points from arcseconds to pixels
    pixel_scale = 0.031  # Assuming 0.031 arcseconds per pixel
    distances = xpos / pixel_scale

    # Calculate radial profiles at the specific distances
    simrad_profile = radial_profile(simimg, simcenter, distances)
    psfrad_profile = radial_profile(psfimg, psfcenter, distances)
    
    # Normalize the profiles
    simrad_profile = simrad_profile / np.max(simrad_profile)
    psfrad_profile = psfrad_profile / np.max(psfrad_profile)

    # Read the CSV file for data
    data = pd.read_csv(datafile, header=None)

    # Extract the sim data
    xdat = data.iloc[:, 0].values  # First column
    ydat = data.iloc[:, 1].values  # Second column
    
    
    if candidate =='JZ11':
        lower_error = [0.008, 0.008, 0.008, ydat[3]-0.124, ydat[4]-0.067, ydat[5]-0.042, ydat[6]-0.0172, ydat[7]-0.0011]
        upper_error = [0.008, 0.008, 0.008, 0.154-ydat[3], 0.093-ydat[4], 0.0664-ydat[5], 0.0388-ydat[6], 0.0195-ydat[7]]
        label = f"Sérsic profile: a = {major:.4f} arcsec, n = {sersic_index:.4f}"
        title = 'JADES-GS-z11 Radial Profile through F200W'
    elif candidate == 'JZ13':
        lower_error = [0.008, 0.008, ydat[2]-0.18, ydat[3]-0.086, ydat[4]-0.038, ydat[5]-0.0067, ydat[6]-0.0011, ydat[7]-0.0018]
        upper_error = [0.008, 0.008, 0.22-ydat[2], 0.12-ydat[3], 0.068-ydat[4], 0.034-ydat[5], 0.024-ydat[6], 0.026-ydat[7]]
        label = f"Sérsic profile: a = {major:.2f} arcsec, n = {sersic_index:.2f}"
        title = 'JADES-GS-z13 Radial Profile through F200W'
    elif candidate == 'JZ14_0':
        lower_error = [0.006, 0.006, 0.006, 0.006, 0.006, 0.006, ydat[6]-0.097, ydat[7]-0.076, ydat[8]-0.0855]
        upper_error = [0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.116-ydat[6], 0.096-ydat[7], 0.108-ydat[8]]
        label = f"Sérsic profile: a = {major:.2f} arcsec, n = {sersic_index:.2f}"
        title = "JADES-GS-z14-0 Radial Profile through F200W"
    elif candidate == 'Maisies':
        lower_error = [0]
        upper_error = [0]
        label = f'2D Gaussian profile: a = {major:.2f} arcsec'
        title = 'Maisies Galaxy Radial Profile through F200W'
    
    mse = mean_squared_error(array_obs, simrad_profile)
    return(mse)
    


In [ ]:
import random
a_array = []
n_array = []
mean_squared = []
n_init_array = []

main_array = np.array(sampler_obs(ydat, lower_error))
for i in range(len(main_array[:,0])):
    n_init = random.gauss(4.0, 0.033)
    n_init_array.append(n_init)
    array_obs = main_array[i,:]
    result = scipy.optimize.minimize(radprof_opt, [0.017, n_init], args = (array_obs, datafile_path, 'JZ13', 'sersic'))
    minimized_val = result.x
    mse_val = result.fun
    print(i)
    print(minimized_val[0], minimized_val[1], mse_val)
    
    a_array.append(minimized_val[0])
    n_array.append(minimized_val[1])
    mean_squared.append(mse_val)

In [ ]:
a_array = np.array(a_array)
n_array = np.array(n_array)
mean_squared = np.array(mean_squared)

print(np.mean(a_array))
print(np.mean(n_array))

In [14]:
def radprof_plot(x, datafile, candidate, shape):
    major = x[0]
    minor = x[0]
    sersic_index = x[1]

    generate_smds_profile(major, minor, sersic_index, candidate, shape)
    simfile = f'{candidate}_f200w_sim.fits'
    psffile = f'{candidate}_f200w_psf.fits'
    # Open FITS files
    simFile = fits.open(simfile)
    simimg = simFile[0].data
#     simimg[np.isnan(simimg)] = 0
    
    psfFile = fits.open(psffile)
    psfimg = psfFile[0].data
#     psfimg[np.isnan(psfimg)] = 0

    # Find centers
    simcenter = np.unravel_index(simimg.argmax(), simimg.shape)
    psfcenter = np.unravel_index(psfimg.argmax(), psfimg.shape)

    # Read the CSV file for comparison points
    df = pd.read_csv(datafile, header=None)
    xpos = df.iloc[:, 0].values  # First column

    # Convert the points from arcseconds to pixels
    pixel_scale = 0.031  # Assuming 0.031 arcseconds per pixel
    distances = xpos / pixel_scale

    # Calculate radial profiles at the specific distances
    simrad_profile = radial_profile(simimg, simcenter, distances)
    psfrad_profile = radial_profile(psfimg, psfcenter, distances)
#     print(simrad_profile)
    # Normalize the profiles
    simrad_profile = simrad_profile / np.max(simrad_profile)
    psfrad_profile = psfrad_profile / np.max(psfrad_profile)

    # Read the CSV file for data
    data = pd.read_csv(datafile, header=None)

    # Extract the sim data
    xdat = data.iloc[:, 0].values  # First column
    ydat = data.iloc[:, 1].values  # Second column
    
    
    if candidate =='JZ11':
        lower_error = [0.008, 0.008, 0.008, ydat[3]-0.124, ydat[4]-0.067, ydat[5]-0.042, ydat[6]-0.0172, ydat[7]-0.0011]
        upper_error = [0.008, 0.008, 0.008, 0.154-ydat[3], 0.093-ydat[4], 0.0664-ydat[5], 0.0388-ydat[6], 0.0195-ydat[7]]
        label = f"Sérsic profile: a = {major:.4f} arcsec, n = {sersic_index:.4f}"
        title = 'JADES-GS-z11 Radial Profile through F200W'
    elif candidate == 'JZ13':
        lower_error = [0.008, 0.008, ydat[2]-0.18, ydat[3]-0.086, ydat[4]-0.038, ydat[5]-0.0067, ydat[6]-0.0011, ydat[7]-0.0018]
        upper_error = [0.008, 0.008, 0.22-ydat[2], 0.12-ydat[3], 0.068-ydat[4], 0.034-ydat[5], 0.024-ydat[6], 0.026-ydat[7]]
        label = f"Sérsic profile: a = {major:.2f} arcsec, n = {sersic_index:.2f}"
        title = 'JADES-GS-z13 Radial Profile through F200W'
    elif candidate == 'JZ14_0':
        lower_error = [0.006, 0.006, 0.006, 0.006, 0.006, 0.006, ydat[6]-0.097, ydat[7]-0.076, ydat[8]-0.0855]
        upper_error = [0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.116-ydat[6], 0.096-ydat[7], 0.108-ydat[8]]
        label = f"Sérsic profile: a = {major:.2f} arcsec, n = {sersic_index:.2f}"
        title = "JADES-GS-z14-0 Radial Profile through F200W"
    elif candidate == 'Maisies':
        lower_error = [0]
        upper_error = [0]
        label = f'2D Gaussian profile: a = {major:.2f} arcsec'
        title = 'Maisies Galaxy Radial Profile through F200W'
    fig, ax = plt.subplots()
    plt.semilogy(xdat, simrad_profile, 'o-', color='magenta', label = label)
    plt.semilogy(xdat, psfrad_profile, 'o-', color='blue', label = 'F200W PSF')
    
    unc = (np.array(upper_error)+np.array(lower_error))/2

    asymmetric_error = np.array(list(zip(lower_error, upper_error))).T

    plt.errorbar(xdat, ydat, yerr = asymmetric_error, fmt='o', color='black', label='JWST data')
    # plt.semilogy(x2, y2, color='lime', label='paper psf')

    ax.xaxis.set_minor_locator(AutoMinorLocator())

    plt.tick_params(which='both', width=1)
    plt.tick_params(which='major', length=7)
    plt.tick_params(which='minor', length=4)
    plt.xlim(0, 0.26)
    plt.grid()
    ax.set_xlabel("Arcseconds")
    ax.set_ylabel("Normalized Radial Profile")
    ax.set_title(title)
    ax.legend(loc='lower left')
    plt.savefig(f'{candidate}_morphology.pdf')
    plt.show()
    

In [ ]:
datafile_path = 'JZ11_rad_prof_data.csv'
radprof_plot([0.101, 4.00], datafile_path, 'JZ11', 'sersic') # replace with mean a_array, n_array